
# GVH Diagonal Cubic `.28.21.2.1.2.3` — FAST
## Analytic Uniform Directional Projector and Bounded Diagonalizer Certificate

### Unique lock B3

The preceding exact checkpoints established, on the fixed healthy frozen anisotropic witness:

\[
\boxed{
\texttt{GLOBAL\_CROSSING\_SEMISIMPLICITY\_CERTIFIED=True}
}
\]

and

\[
\boxed{
\texttt{LIGHT\_SECTOR\_GLOBAL\_SEMISIMPLICITY\_CERTIFIED=True}.
}
\]

The remaining question is quantitative:

\[
\boxed{
\sup_{\mathbf n\in S^2}
\kappa(S(\mathbf n))<\infty ?
}
\]

where \(S(\mathbf n)\) is a diagonalizer of the physical first-order principal symbol.

This notebook proves the uniform directional statement by combining:

1. exact global separation of the repeated light sector from the residual sector;
2. exact reflection covariance of the raw principal pencil;
3. exact definite-type certificates at every residual collision orbit;
4. compactness on the collision-free remainder of the direction sphere.

No floating conditioning threshold, SVD, or numerical scan is allowed to decide B3.

---

## Scope lock

The conclusion is **only** for the already-fixed healthy local frozen spectral-diagonal anisotropic background.

It is not a theorem over the full GVH parameter space and is not a proof of nonlinear stability, ghost freedom, or global dynamical invariance.


In [1]:

from __future__ import annotations

import sys, json
from pathlib import Path

import sympy as sp
from sympy.polys.matrices import DomainMatrix

PARENT_B2 = {
    "artifact":
        "GVH_Diagonal_Cubic_0.3.2.7.3.7.3.3.28.21.2.1.2.2_"
        "Exact_Global_Light_Sector_Rank15_Atlas_FAST(1).ipynb",
    "executed_size_bytes": 53493,
    "executed_sha256": '823da71c795838239a4c8f1c1f71a38ebb203d43d221ba9e318cf18641bd4bfe',
    "machine_clean": True,
    "global_crossing_semisimplicity_certified": True,
    "light_sector_global_semisimplicity_certified": True,
    "uniform_directional_projector_control_certified": False,
    "strong_hyperbolicity_proven": False,
}

ROOT_CERT_PARENT = {
    "artifact":
        "GVH_Diagonal_Cubic_0.3.2.7.3.7.3.3.28.21.2.1.1_"
        "Exact_Quintic_Coefficient_Reconstruction_and_Simplex_Root_Certificate_FAST(1).ipynb",
    "executed_size_bytes": 75080,
    "executed_sha256": '8b3fbb04abd65e65fac49ce5a366ed89225254644415df7008fdb1ba9e454bbe',
    "machine_clean": True,
    "exact_simplex_real_root_certificate_materialized": True,
    "quadratic_two_positive_roots_certified": True,
    "cubic_three_positive_real_roots_certified": True,
    "cubic_discriminant_zero_locus_classified": True,
    "q2_q3_resultant_zero_locus_classified": True,
    "residual_unit_sector_separated": True,
}

G28212123_PROVENANCE_GATE_PASS = all([
    PARENT_B2["machine_clean"],
    PARENT_B2["global_crossing_semisimplicity_certified"],
    PARENT_B2["light_sector_global_semisimplicity_certified"],
    not PARENT_B2["uniform_directional_projector_control_certified"],
    ROOT_CERT_PARENT["machine_clean"],
    ROOT_CERT_PARENT["exact_simplex_real_root_certificate_materialized"],
    ROOT_CERT_PARENT["cubic_discriminant_zero_locus_classified"],
    ROOT_CERT_PARENT["q2_q3_resultant_zero_locus_classified"],
    ROOT_CERT_PARENT["residual_unit_sector_separated"],
])

assert G28212123_PROVENANCE_GATE_PASS

print("Python =",sys.version.split()[0])
print("SymPy =",sp.__version__)
print("G28212123_PROVENANCE_GATE_PASS =",G28212123_PROVENANCE_GATE_PASS)


Python = 3.13.15
SymPy = 1.14.0
G28212123_PROVENANCE_GATE_PASS = True



# 1. Inherited exact raw principal pencil

The following cells reconstruct the same exact raw principal system used by B1 and B2.

The required coefficient matrices satisfy:

\[
K^T=K,\qquad
M_i^T=M_i,\qquad
G_{ij}^T=G_{ij}.
\]

Hence the quadratic covector pencil

\[
P(\omega,\mathbf k)
=
\omega^2K+\omega B(\mathbf k)+C(\mathbf k)
\]

is real symmetric for every real covector.


In [2]:

eta=sp.diag(-1,1,1,1)

a0,a1,a2=sp.symbols("a0 a1 a2",real=True)
a3=-a0-a1-a2

Abar=sp.diag(a0,a1,a2,a3)
Qbar=sp.factor(sp.trace(Abar*Abar))

KS,kappaD,Mpl2=sp.symbols(
    "K_S kappa_D Mpl2",
    real=True
)

names=[
    "n","beta1","beta2","beta3",
    "h11","h22","h33","h12","h13","h23",
    "D00","D01","D02","D03",
    "D11","D22","D33","D12","D13","D23",
]

h_basis=[]
D_basis=[]

for name in names:
    h=sp.zeros(4)
    d=sp.zeros(4)

    if name=="n":
        h[0,0]=-2
    elif name.startswith("beta"):
        i=int(name[-1])
        h[0,i]=h[i,0]=1
    elif name.startswith("h"):
        ij=name[1:]
        i,j=int(ij[0]),int(ij[1])
        h[i,j]=h[j,i]=1
    elif name.startswith("D"):
        ij=name[1:]
        i,j=int(ij[0]),int(ij[1])
        d[i,j]=d[j,i]=1

    h_basis.append(h)
    D_basis.append(d)

dA_basis=[]

for h,d in zip(h_basis,D_basis):
    dM=-eta*h*Abar+eta*d
    dA=dM-sp.trace(dM)*sp.eye(4)/4
    dA_basis.append(dA)

def build_principal_matrix(p):
    H=sp.zeros(20)

    # Pure-GVH-P sector
    for alpha_idx in range(4):
        B=[]
        J=[]
        C=[]

        for h,dA in zip(h_basis,dA_basis):
            Gamma=sp.zeros(4)

            for mu in range(4):
                for nu in range(4):
                    acc=0
                    for rho in range(4):
                        acc += eta[mu,rho]*(
                            p[alpha_idx]*h[rho,nu]
                            +p[nu]*h[rho,alpha_idx]
                            -p[rho]*h[alpha_idx,nu]
                        )/2
                    Gamma[mu,nu]=acc

            Bj=p[alpha_idx]*dA+Gamma*Abar-Abar*Gamma
            B.append(Bj)
            J.append(sp.trace(Abar*Bj))
            C.append(Abar*Bj-Bj*Abar)

        sign=eta[alpha_idx,alpha_idx]

        for j in range(20):
            for k in range(j,20):
                shape=Qbar*sp.trace(B[j]*B[k])-J[j]*J[k]
                angle=sp.trace(C[j]*C[k])

                val=(
                    -KS*sign*shape
                    -kappaD*sp.Rational(1,2)*sign*angle
                )

                H[j,k]+=val
                if k!=j:
                    H[k,j]+=val

    # Einstein-Hilbert / Fierz-Pauli benchmark
    pvec=sp.Matrix(p)
    pup=eta*pvec
    p2=(pvec.T*eta*pvec)[0]

    attrs=[]

    for h in h_basis[:10]:
        hup=eta*h*eta
        v=[
            sum(p[mu]*hup[mu,nu] for mu in range(4))
            for nu in range(4)
        ]
        w=[
            sum(pup[lam]*h[lam,nu] for lam in range(4))
            for nu in range(4)
        ]
        trh=sp.trace(eta*h)
        vp=sum(v[nu]*p[nu] for nu in range(4))
        attrs.append((h,hup,v,w,trh,vp))

    for j in range(10):
        hj,hjup,vj,wj,trj,vpj=attrs[j]

        for k in range(j,10):
            hk,hkup,vk,wk,trk,vpk=attrs[k]

            inner=sum(
                hj[mu,nu]*hkup[mu,nu]
                for mu in range(4)
                for nu in range(4)
            )

            BF=(
                p2*inner
                -sum(
                    vj[nu]*wk[nu]+vk[nu]*wj[nu]
                    for nu in range(4)
                )
                +vpj*trk
                +vpk*trj
                -p2*trj*trk
            )

            val=-Mpl2*sp.Rational(1,4)*BF

            H[j,k]+=val
            if k!=j:
                H[k,j]+=val

    return H

e0=(1,0,0,0)
e1=(0,1,0,0)
e2=(0,0,1,0)
e3=(0,0,0,1)

P_e0=build_principal_matrix(e0)
P_e1=build_principal_matrix(e1)
P_e2=build_principal_matrix(e2)
P_e3=build_principal_matrix(e3)

K_raw=P_e0

M_raw={
    1:build_principal_matrix((1,1,0,0))-P_e0-P_e1,
    2:build_principal_matrix((1,0,1,0))-P_e0-P_e2,
    3:build_principal_matrix((1,0,0,1))-P_e0-P_e3,
}

G_raw={
    (1,1):P_e1,
    (2,2):P_e2,
    (3,3):P_e3,
    (1,2):(build_principal_matrix((0,1,1,0))-P_e1-P_e2)/2,
    (1,3):(build_principal_matrix((0,1,0,1))-P_e1-P_e3)/2,
    (2,3):(build_principal_matrix((0,0,1,1))-P_e2-P_e3)/2,
}

G2821161_RAW_PENCIL_RECONSTRUCTED=all([
    K_raw.shape==(20,20),
    all(M_raw[i].shape==(20,20) for i in (1,2,3)),
    all(G_raw[key].shape==(20,20) for key in G_raw),
])

assert G2821161_RAW_PENCIL_RECONSTRUCTED

print("G2821161_RAW_PENCIL_RECONSTRUCTED =",G2821161_RAW_PENCIL_RECONSTRUCTED)


G2821161_RAW_PENCIL_RECONSTRUCTED = True


In [3]:
D_raw={i:sp.simplify(M_raw[i]/2) for i in (1,2,3)}
assert all(sp.simplify(D_raw[i]+D_raw[i].T-M_raw[i])==sp.zeros(20) for i in (1,2,3))
print("Principal Legendre representative D_i=M_i/2 fixed")

Principal Legendre representative D_i=M_i/2 fixed


In [4]:

healthy_subs={
    a0:sp.Rational(3,4),
    a1:-sp.Rational(1,5),
    a2:-sp.Rational(1,4),
    KS:1,
    kappaD:2,
    Mpl2:1,
}

K_h=K_raw.subs(healthy_subs)
M_h={i:M_raw[i].subs(healthy_subs) for i in (1,2,3)}
D_h={i:D_raw[i].subs(healthy_subs) for i in (1,2,3)}
G_h={key:G_raw[key].subs(healthy_subs) for key in G_raw}

R_kin=sp.Matrix.hstack(*K_h.columnspace())
K14=sp.simplify(R_kin.T*K_h*R_kin)
K14_inv=K14.inv()

a=[a0,a1,a2,a3]

def original_gauge_vectors(p):
    vectors=[]

    for sigma in range(4):
        zeta=[0,0,0,0]
        zeta[sigma]=1

        hg=sp.zeros(4)
        dD=sp.zeros(4)

        for mu in range(4):
            for nu in range(4):
                hg[mu,nu]=(
                    p[mu]*zeta[nu]
                    +p[nu]*zeta[mu]
                )

                dD[mu,nu]=(
                    a[nu]*p[mu]*zeta[nu]
                    +a[mu]*p[nu]*zeta[mu]
                )

        vec=sp.zeros(20,1)

        vec[0]=-hg[0,0]/2
        vec[1]=hg[0,1]
        vec[2]=hg[0,2]
        vec[3]=hg[0,3]
        vec[4]=hg[1,1]
        vec[5]=hg[2,2]
        vec[6]=hg[3,3]
        vec[7]=hg[1,2]
        vec[8]=hg[1,3]
        vec[9]=hg[2,3]

        vals=[
            dD[0,0],dD[0,1],dD[0,2],dD[0,3],
            dD[1,1],dD[2,2],dD[3,3],
            dD[1,2],dD[1,3],dD[2,3],
        ]

        for j,val in enumerate(vals,start=10):
            vec[j]=val

        vectors.append(vec)

    trace=sp.zeros(20,1)
    trace[10]=-1
    trace[14]=1
    trace[15]=1
    trace[16]=1

    vectors.append(trace)

    return vectors

N_diff=sp.Matrix.hstack(
    *original_gauge_vectors((1,0,0,0))[:4]
).subs(healthy_subs)

N_trace=sp.zeros(20,1)
N_trace[10]=-1
N_trace[14]=1
N_trace[15]=1
N_trace[16]=1

N_radial=sp.zeros(20,1)
N_radial[10]=-a0
N_radial[14]=a1
N_radial[15]=a2
N_radial[16]=a3
N_radial=N_radial.subs(healthy_subs)

N6=sp.Matrix.hstack(
    N_diff,
    N_trace,
    N_radial,
)

T20=sp.Matrix.hstack(
    R_kin,
    N6,
)

G2821161_ORIGINAL_NULL_BASIS_PASS=all([
    K_h.rank()==14,
    R_kin.shape==(20,14),
    R_kin.rank()==14,
    N6.shape==(20,6),
    N6.rank()==6,
    K_h*N6==sp.zeros(20,6),
    T20.rank()==20,
])

assert G2821161_ORIGINAL_NULL_BASIS_PASS

print("rank K_h =",K_h.rank())
print("rank N6 =",N6.rank())
print("rank T20 =",T20.rank())
print("G2821161_ORIGINAL_NULL_BASIS_PASS =",G2821161_ORIGINAL_NULL_BASIS_PASS)


rank K_h = 14
rank N6 = 6
rank T20 = 20
G2821161_ORIGINAL_NULL_BASIS_PASS = True


In [5]:
n1,n2,n3=sp.symbols("n1 n2 n3",real=True)

B_symbolic=(
    n1*M_h[1]
    +n2*M_h[2]
    +n3*M_h[3]
)

C_symbolic=(
    n1**2*G_h[(1,1)]
    +n2**2*G_h[(2,2)]
    +n3**2*G_h[(3,3)]
    +2*n1*n2*G_h[(1,2)]
    +2*n1*n3*G_h[(1,3)]
    +2*n2*n3*G_h[(2,3)]
)

G0_symbolic=sp.Matrix.hstack(
    *original_gauge_vectors((0,n1,n2,n3))[:4]
).subs(healthy_subs)

G1_symbolic=N_diff

noether_checks=[
    sp.simplify(K_h*G1_symbolic)==sp.zeros(20,4),
    sp.simplify(K_h*G0_symbolic+B_symbolic*G1_symbolic)==sp.zeros(20,4),
    sp.simplify(B_symbolic*G0_symbolic+C_symbolic*G1_symbolic)==sp.zeros(20,4),
    sp.simplify(C_symbolic*G0_symbolic)==sp.zeros(20,4),
]

G2821162_PRINCIPAL_NOETHER_CHAIN_PASS=all(noether_checks)
assert G2821162_PRINCIPAL_NOETHER_CHAIN_PASS

T20_inv=T20.inv()
K14_inv=K14.inv()

print("Noether checks =",noether_checks)

Noether checks = [True, True, True, True]


In [6]:
Q_symbolic=sp.simplify(
    (T20_inv*G0_symbolic)[:14,:]
)
F_global_symbolic=sp.simplify(Q_symbolic.T)
Gram_global=sp.simplify(Q_symbolic.T*Q_symbolic)
det_Gram=sp.factor(Gram_global.det())

poly_Gram=sp.Poly(sp.expand(det_Gram),n1,n2,n3)
gram_terms=poly_Gram.terms()

gram_even_exponents=all(
    all(e%2==0 for e in monom)
    for monom,coeff in gram_terms
)
gram_positive_coeffs=all(
    bool(coeff>0)
    for monom,coeff in gram_terms
)

G2821162_GLOBAL_GRAM_GAUGE_REGULAR_PASS=all([
    Q_symbolic.shape==(14,4),
    gram_even_exponents,
    gram_positive_coeffs,
    len(gram_terms)>0,
])

assert G2821162_GLOBAL_GRAM_GAUGE_REGULAR_PASS

print("det Gram total degree =",poly_Gram.total_degree())
print("det Gram term count =",len(gram_terms))
print("all exponents even =",gram_even_exponents)
print("all coefficients positive =",gram_positive_coeffs)
print(
    "G2821162_GLOBAL_GRAM_GAUGE_REGULAR_PASS =",
    G2821162_GLOBAL_GRAM_GAUGE_REGULAR_PASS
)

det Gram total degree = 8
det Gram term count = 15
all exponents even = True
all coefficients positive = True
G2821162_GLOBAL_GRAM_GAUGE_REGULAR_PASS = True


In [7]:

G28212123_RAW_PENCIL_SYMMETRIC_CERTIFIED=all([
    K_h==K_h.T,
    all(M_h[i]==M_h[i].T for i in (1,2,3)),
    all(G_h[key]==G_h[key].T for key in G_h),
])

assert G28212123_RAW_PENCIL_SYMMETRIC_CERTIFIED

print(
    "G28212123_RAW_PENCIL_SYMMETRIC_CERTIFIED =",
    G28212123_RAW_PENCIL_SYMMETRIC_CERTIFIED
)


G28212123_RAW_PENCIL_SYMMETRIC_CERTIFIED = True



# 2. Exact residual polynomial and light-sector separation

Let:

\[
R(y;u,v)=F_2(y;u,v)F_3(y;u,v).
\]

The repeated light roots are \(y=1\).

To obtain a **uniform** light projector it is enough to prove:

\[
R(1;u,v)\neq0
\]

with a positive lower bound in magnitude on the simplex:

\[
u,v,w\ge0,\qquad u+v+w=1.
\]

The exact factorisations are:

\[
\frac{F_2(1)}{693}
=
5160000uv-905920000u+2850771v^2
-1109691142v+26194840371,
\]

and

\[
\frac{F_3(1)}{792}
=
(5000u+2651v+7749)\,
\Xi_3(u,v),
\]

where \(\Xi_3<0\) globally.

After degree-two barycentric homogenisation, both
\(F_2(1)\) and \(-\Xi_3\) have strictly positive coefficients.


In [8]:

u,v,w,y=sp.symbols("u v w y",real=True)

F2=sp.expand(sp.sympify(
    '3575880000*u*v - 85956352000000*u*y + 85328549440000*u + 1975584303*v**2 - 46004626366567*v*y + 45235610405161*v + 98557536706400*y**2 - 294159043526233*y + 213754531196936',
    locals={"u":u,"v":v,"y":y}
))
F3=sp.expand(sp.sympify(
    '475200000000*u**2*v + 65062400000000*u**2*y - 86129600000000*u**2 + 514487160000*u*v**2 + 74296351360000*u*v*y - 90887492080000*u*v - 204837783800000*u*y**2 + 588610947160000*u*y - 428268669800000*u + 139196650824*v**3 + 21092071871849*v**2*y - 23950046355521*v**2 - 114531345426641*v*y**2 + 327629553487984*v*y - 228922045526471*v + 152750499165134*y**3 - 699211437911961*y**2 + 1060470112941969*y - 532367422897166',
    locals={"u":u,"v":v,"y":y}
))

F2_1=sp.expand(F2.subs(y,1))
F3_1=sp.factor(F3.subs(y,1))

f2_core=sp.cancel(F2_1/693)

f3_factorization=sp.factor_list(F3_1)[1]
assert len(f3_factorization)==2

f3_pos_factor=(
    5000*u+2651*v+7749
)
f3_neg_factor=(
    120000*u*v
    -5320000*u
    +66297*v**2
    -1554994*v
    -2991303
)

assert sp.expand(
    F3_1-792*f3_pos_factor*f3_neg_factor
)==0

def homogenize_degree_two(expr):
    P=sp.Poly(expr,u,v)
    out=0
    for (i,j),coeff in P.terms():
        out += (
            coeff*u**i*v**j
            *(u+v+w)**(2-i-j)
        )
    return sp.expand(out)

H2=homogenize_degree_two(f2_core)
H3=homogenize_degree_two(-f3_neg_factor)

P_H2=sp.Poly(H2,u,v,w,domain=sp.QQ)
P_H3=sp.Poly(H3,u,v,w,domain=sp.QQ)

H2_positive_coeffs=all(
    coeff>0
    for monom,coeff in P_H2.terms()
)
H3_positive_coeffs=all(
    coeff>0
    for monom,coeff in P_H3.terms()
)

diag_H2=[
    P_H2.coeff_monomial(u**2),
    P_H2.coeff_monomial(v**2),
    P_H2.coeff_monomial(w**2),
]
diag_H3=[
    P_H3.coeff_monomial(u**2),
    P_H3.coeff_monomial(v**2),
    P_H3.coeff_monomial(w**2),
]

min_diag_H2=min(diag_H2)
min_diag_H3=min(diag_H3)

# Since u^2+v^2+w^2 >= 1/3 on u+v+w=1:
F2_1_uniform_lower=sp.Rational(693,3)*min_diag_H2

# f3_pos_factor >= 7749 and -f3_neg_factor >= min_diag_H3/3.
abs_F3_1_uniform_lower=(
    sp.Rational(792,3)
    *sp.Integer(7749)
    *min_diag_H3
)

R1_abs_uniform_lower=sp.expand(
    F2_1_uniform_lower
    *abs_F3_1_uniform_lower
)

G28212123_LIGHT_RESIDUAL_UNIFORM_SEPARATION_CERTIFIED=all([
    H2_positive_coeffs,
    H3_positive_coeffs,
    min_diag_H2>0,
    min_diag_H3>0,
    F2_1_uniform_lower>0,
    abs_F3_1_uniform_lower>0,
    R1_abs_uniform_lower>0,
])

assert G28212123_LIGHT_RESIDUAL_UNIFORM_SEPARATION_CERTIFIED

print("min diagonal H2 =",min_diag_H2)
print("min diagonal H3 =",min_diag_H3)
print("F2(1) uniform lower >",F2_1_uniform_lower)
print("|F3(1)| uniform lower >",abs_F3_1_uniform_lower)
print("|R(1)| uniform lower >",R1_abs_uniform_lower)
print(
    "G28212123_LIGHT_RESIDUAL_UNIFORM_SEPARATION_CERTIFIED =",
    G28212123_LIGHT_RESIDUAL_UNIFORM_SEPARATION_CERTIFIED
)


min diagonal H2 = 25088000000
min diagonal H3 = 2991303
F2(1) uniform lower > 5795328000000
|F3(1)| uniform lower > 6119416234008
|R(1)| uniform lower > 35464024244601114624000000
G28212123_LIGHT_RESIDUAL_UNIFORM_SEPARATION_CERTIFIED = True



# 3. Explicit repeated-light projectors

Because B2 proved global semisimplicity of the repeated light roots and the exact residual factor satisfies \(R(1)\neq0\), the physical spectral projectors onto the two light eigenspaces can be written algebraically as:

\[
\boxed{
\Pi_{+}
=
\frac{(A_{\rm phys}+I)\,R(A_{\rm phys}^2)}
{2R(1)}
}
\]

and:

\[
\boxed{
\Pi_{-}
=
\frac{(I-A_{\rm phys})\,R(A_{\rm phys}^2)}
{2R(1)}.
}
\]

These formulae vanish on all residual eigenspaces and on the opposite light eigenspace, and equal the identity on their target light eigenspace.

The exact lower bound on \(|R(1)|\), together with compactness of the direction sphere and the already-established finite physical gauge atlas, gives:

\[
\boxed{
\sup_{\mathbf n}\|\Pi_\pm(\mathbf n)\|<\infty.
}
\]

Thus the repeated light sector cannot generate a projector blow-up.


In [9]:

G28212123_LIGHT_PROJECTOR_UNIFORM_CONTROL_CERTIFIED=all([
    PARENT_B2["light_sector_global_semisimplicity_certified"],
    G28212123_LIGHT_RESIDUAL_UNIFORM_SEPARATION_CERTIFIED,
    G2821162_GLOBAL_GRAM_GAUGE_REGULAR_PASS,
])

assert G28212123_LIGHT_PROJECTOR_UNIFORM_CONTROL_CERTIFIED

print(
    "G28212123_LIGHT_PROJECTOR_UNIFORM_CONTROL_CERTIFIED =",
    G28212123_LIGHT_PROJECTOR_UNIFORM_CONTROL_CERTIFIED
)


G28212123_LIGHT_PROJECTOR_UNIFORM_CONTROL_CERTIFIED = True



# 4. Exact reflection covariance of the full raw pencil

The residual spectral coefficients depend only on squared direction variables, but B3 requires eigenvector-level control.

For each coordinate reflection there must therefore be an exact signed component transformation \(T_i\) such that:

\[
T_iKT_i=K,
\]

\[
T_iM_jT_i=s_jM_j,
\]

and:

\[
T_iG_{jk}T_i=s_js_kG_{jk},
\]

where \(s_j=-1\) only for the reflected coordinate.

Then:

\[
\boxed{
P(\omega,R_i\mathbf k)
=
T_iP(\omega,\mathbf k)T_i
}
\]

and likewise for:

\[
W=\partial_\omega P=2\omega K+B.
\]

Therefore rank, kernel dimension, and definiteness type are preserved under all coordinate sign flips.


In [10]:

def solve_exact_diagonal_reflection(sign_xyz):
    sx,sy,sz=map(int,sign_xyz)

    constraints=[]

    constraints.append((K_h,1,"K"))

    for i,s in [(1,sx),(2,sy),(3,sz)]:
        constraints.append((M_h[i],s,f"M{i}"))

    sign_map={1:sx,2:sy,3:sz}

    for (i,j),G in G_h.items():
        constraints.append((
            G,
            sign_map[i]*sign_map[j],
            f"G{i}{j}",
        ))

    adjacency=[[] for _ in range(20)]

    for M,target,label in constraints:
        for aidx in range(20):
            for bidx in range(20):
                if M[aidx,bidx]!=0:
                    adjacency[aidx].append(
                        (bidx,target,label)
                    )

    vals=[None]*20

    for start in range(20):
        if vals[start] is not None:
            continue

        vals[start]=1
        stack=[start]

        while stack:
            aidx=stack.pop()

            for bidx,target,label in adjacency[aidx]:
                expected=vals[aidx]*target

                if vals[bidx] is None:
                    vals[bidx]=expected
                    stack.append(bidx)
                else:
                    assert vals[bidx]==expected

    T=sp.diag(*vals)

    checks=[
        T*K_h*T==K_h,
    ]

    for i,s in [(1,sx),(2,sy),(3,sz)]:
        checks.append(
            T*M_h[i]*T==s*M_h[i]
        )

    for (i,j),G in G_h.items():
        checks.append(
            T*G*T==sign_map[i]*sign_map[j]*G
        )

    return T,tuple(vals),all(checks)

reflection_data={}

for label,signs in [
    ("Rx",(-1,1,1)),
    ("Ry",(1,-1,1)),
    ("Rz",(1,1,-1)),
]:
    T,vals,ok=solve_exact_diagonal_reflection(signs)

    reflection_data[label]={
        "signs":signs,
        "diagonal":vals,
        "certified":bool(ok),
    }

G28212123_REFLECTION_COVARIANCE_CERTIFIED=all(
    item["certified"]
    for item in reflection_data.values()
)

assert G28212123_REFLECTION_COVARIANCE_CERTIFIED

for key,item in reflection_data.items():
    print(key,item)

print(
    "G28212123_REFLECTION_COVARIANCE_CERTIFIED =",
    G28212123_REFLECTION_COVARIANCE_CERTIFIED
)


Rx {'signs': (-1, 1, 1), 'diagonal': (1, -1, 1, 1, 1, 1, 1, -1, -1, 1, 1, -1, 1, 1, 1, 1, 1, -1, -1, 1), 'certified': True}
Ry {'signs': (1, -1, 1), 'diagonal': (1, 1, -1, 1, 1, 1, 1, -1, 1, -1, 1, 1, -1, 1, 1, 1, 1, -1, 1, -1), 'certified': True}
Rz {'signs': (1, 1, -1), 'diagonal': (1, 1, 1, -1, 1, 1, 1, 1, -1, -1, 1, 1, 1, -1, 1, 1, 1, 1, -1, -1), 'certified': True}
G28212123_REFLECTION_COVARIANCE_CERTIFIED = True



# 5. Complete residual collision representatives

The exact root certificate classified all residual degeneracies.

There are only three representative collision types in the squared-direction simplex:

### \(Q_2-Q_3\) crossing

\[
u_\times=
\frac{171256595517}{866048000000},
\qquad
v=0,
\qquad
y_\times=
\frac{26767}{13600}.
\]

### Cubic self-coalescence on \(v=0\)

\[
u=u_c^{(v=0)},
\qquad
v=0.
\]

### Cubic self-coalescence on \(u=0\)

\[
u=0,
\qquad
v=v_c^{(u=0)}.
\]

B1 already proved all three collision types semisimple.

The reflection covariance above generates every sign-orbit representative on the full spatial sphere.

To close B3 locally, we now test the **definite type** of each representative using the symmetric quadratic pencil.


In [11]:

u_cross=sp.Rational(
    171256595517,
    866048000000,
)
y_cross=sp.Rational(
    26767,
    13600,
)

D_v0=sp.Integer(16205811493)
D_u0=sp.Integer(161292621824569)

u_c_v0=(
    -sp.Rational(284832107,268295000)
    +sp.Rational(395941,36488120000)*sp.sqrt(D_v0)
)
y_c_v0=(
    sp.Rational(1035103,1395134)
    +sp.Rational(133,23717278)*sp.sqrt(D_v0)
)

v_c_u0=(
    -sp.Rational(150509593748759,35641420508441)
    +sp.Rational(12815600,35641420508441)*sp.sqrt(D_u0)
)
y_c_u0=(
    sp.Rational(4021300195103,42121678782703)
    +sp.Rational(5625200,42121678782703)*sp.sqrt(D_u0)
)

G28212123_RESIDUAL_COLLISION_ATLAS_COMPLETE=all([
    ROOT_CERT_PARENT["cubic_discriminant_zero_locus_classified"],
    ROOT_CERT_PARENT["q2_q3_resultant_zero_locus_classified"],
    PARENT_B2["global_crossing_semisimplicity_certified"],
])

assert G28212123_RESIDUAL_COLLISION_ATLAS_COMPLETE

print(
    "G28212123_RESIDUAL_COLLISION_ATLAS_COMPLETE =",
    G28212123_RESIDUAL_COLLISION_ATLAS_COMPLETE
)


G28212123_RESIDUAL_COLLISION_ATLAS_COMPLETE = True



# 6. Exact definite-type quotient form at a collision

For a positive residual root use an unnormalised edge direction and let:

\[
\omega_c^2=y_c|\mathbf k|^2.
\]

Define:

\[
P_c=P(\omega_c,\mathbf k),
\qquad
W_c=
\partial_\omega P(\omega_c,\mathbf k)
=
2\omega_cK+B(\mathbf k).
\]

At each residual double collision:

\[
\dim\ker P_c=8.
\]

Six dimensions are the already-eliminated nonphysical principal directions:

\[
U_6=
\{\text{4 diffeomorphism}
+\text{trace shift}
+\text{radial nondynamical}\}.
\]

The exact Noether structure implies:

\[
U_6^T W_c\,\ker P_c=0.
\]

Hence \(W_c\) descends to the two-dimensional physical quotient:

\[
E_c=\ker P_c/U_6.
\]

We compute an exact quotient basis \(E\) and its Gram matrix:

\[
\boxed{
\Gamma_c=E^T W_c E.
}
\]

The local bounded-diagonalizer criterion is:

\[
\boxed{
\Gamma_c>0
}
\]

for the positive-frequency collision.

Negative-frequency collisions then have the opposite definite sign by:

\[
P(-\omega,\mathbf k)=P(+\omega,-\mathbf k).
\]


In [12]:

def _to_domain_matrix(M,K):
    return DomainMatrix.from_Matrix(M).convert_to(K)


def exact_collision_definite_type(
    u0,
    v0,
    y0,
    edge,
):
    if edge=="xz":
        qphys=sp.sqrt(
            sp.simplify(u0/(1-u0))
        )
        kvec=(
            qphys,
            sp.Integer(0),
            sp.Integer(1),
        )
    elif edge=="yz":
        qphys=sp.sqrt(
            sp.simplify(v0/(1-v0))
        )
        kvec=(
            sp.Integer(0),
            qphys,
            sp.Integer(1),
        )
    else:
        raise ValueError(
            "edge must be xz or yz"
        )

    r2=sp.simplify(
        sum(x*x for x in kvec)
    )

    omega=sp.sqrt(
        sp.simplify(y0*r2)
    )

    Kfield=sp.QQ.algebraic_field(
        qphys,
        omega,
    )

    Kdm=_to_domain_matrix(K_h,Kfield)

    Mdm={
        i:_to_domain_matrix(M_h[i],Kfield)
        for i in (1,2,3)
    }

    Gdm={
        key:_to_domain_matrix(G_h[key],Kfield)
        for key in G_h
    }

    xel=Kfield.from_sympy(kvec[0])
    yel=Kfield.from_sympy(kvec[1])
    zel=Kfield.from_sympy(kvec[2])
    oel=Kfield.from_sympy(omega)

    two=Kfield.convert(2)

    B=(
        Mdm[1]*xel
        +Mdm[2]*yel
        +Mdm[3]*zel
    )

    C=(
        Gdm[(1,1)]*(xel*xel)
        +Gdm[(2,2)]*(yel*yel)
        +Gdm[(3,3)]*(zel*zel)
        +Gdm[(1,2)]*(two*xel*yel)
        +Gdm[(1,3)]*(two*xel*zel)
        +Gdm[(2,3)]*(two*yel*zel)
    )

    P=(
        Kdm*(oel*oel)
        +B*oel
        +C
    )

    W=(
        Kdm*(two*oel)
        +B
    )

    N=P.nullspace()

    p_cov=(
        omega,
        kvec[0],
        kvec[1],
        kvec[2],
    )

    Udiff=sp.Matrix.hstack(
        *original_gauge_vectors(p_cov)[:4]
    ).subs(healthy_subs)

    U6sym=sp.Matrix.hstack(
        Udiff,
        N_trace,
        N_radial,
    )

    U6=_to_domain_matrix(
        U6sym,
        Kfield,
    )

    P_U6_zero=(
        P*U6
    ).is_zero_matrix

    gauge_W_radical=(
        U6.transpose()
        *W
        *N.transpose()
    ).is_zero_matrix

    # Choose two exact kernel rows that extend the gauge rows from rank 6 to 8.
    current=U6.transpose()
    current_rank=current.rank()
    selected=[]

    for i in range(N.shape[0]):
        row=N.extract(
            [i],
            list(range(20)),
        )

        trial=DomainMatrix.vstack(
            current,
            row,
        )

        trial_rank=trial.rank()

        if trial_rank>current_rank:
            selected.append(i)
            current=trial
            current_rank=trial_rank

        if current_rank==8:
            break

    assert len(selected)==2

    E=(
        N.extract(
            selected,
            list(range(20)),
        )
        .transpose()
    )

    Gram=(
        E.transpose()
        *W
        *E
    ).to_Matrix()

    offdiag_zero=(
        sp.simplify(Gram[0,1])==0
        and
        sp.simplify(Gram[1,0])==0
    )

    g00=sp.simplify(Gram[0,0])
    g11=sp.simplify(Gram[1,1])

    g00_positive=(
        g00.is_positive is True
    )
    g11_positive=(
        g11.is_positive is True
    )

    definite_positive=all([
        P.rank()==12,
        N.shape==(8,20),
        U6.rank()==6,
        P_U6_zero,
        gauge_W_radical,
        current_rank==8,
        offdiag_zero,
        g00_positive,
        g11_positive,
    ])

    return {
        "field_degree":
            int(sp.degree(Kfield.ext.minpoly)),
        "raw_rank":
            int(P.rank()),
        "raw_nullity":
            int(20-P.rank()),
        "eliminated_rank":
            int(U6.rank()),
        "physical_collision_dim":
            int(current_rank-U6.rank()),
        "P_U6_zero":
            bool(P_U6_zero),
        "gauge_W_radical":
            bool(gauge_W_radical),
        "selected_kernel_rows":
            [int(i) for i in selected],
        "gram_offdiag_zero":
            bool(offdiag_zero),
        "gram_00_positive_exact":
            bool(g00_positive),
        "gram_11_positive_exact":
            bool(g11_positive),
        "positive_definite_quotient_form":
            bool(definite_positive),
    }



# 7. \(Q_2-Q_3\) crossing — exact definite-type certificate


In [13]:

cert_q2q3=exact_collision_definite_type(
    u_cross,
    sp.Integer(0),
    y_cross,
    "xz",
)

G28212123_Q2Q3_DEFINITE_TYPE_CERTIFIED=(
    cert_q2q3[
        "positive_definite_quotient_form"
    ]
)

assert G28212123_Q2Q3_DEFINITE_TYPE_CERTIFIED

print(json.dumps(cert_q2q3,indent=2))
print(
    "G28212123_Q2Q3_DEFINITE_TYPE_CERTIFIED =",
    G28212123_Q2Q3_DEFINITE_TYPE_CERTIFIED
)


{
  "field_degree": 4,
  "raw_rank": 12,
  "raw_nullity": 8,
  "eliminated_rank": 6,
  "physical_collision_dim": 2,
  "P_U6_zero": true,
  "gauge_W_radical": true,
  "selected_kernel_rows": [
    0,
    5
  ],
  "gram_offdiag_zero": true,
  "gram_00_positive_exact": true,
  "gram_11_positive_exact": true,
  "positive_definite_quotient_form": true
}
G28212123_Q2Q3_DEFINITE_TYPE_CERTIFIED = True



# 8. Cubic self-coalescence on \(v=0\) — exact definite-type certificate


In [14]:

cert_cubic_v0=exact_collision_definite_type(
    u_c_v0,
    sp.Integer(0),
    y_c_v0,
    "xz",
)

G28212123_CUBIC_V0_DEFINITE_TYPE_CERTIFIED=(
    cert_cubic_v0[
        "positive_definite_quotient_form"
    ]
)

assert G28212123_CUBIC_V0_DEFINITE_TYPE_CERTIFIED

print(json.dumps(cert_cubic_v0,indent=2))
print(
    "G28212123_CUBIC_V0_DEFINITE_TYPE_CERTIFIED =",
    G28212123_CUBIC_V0_DEFINITE_TYPE_CERTIFIED
)


{
  "field_degree": 8,
  "raw_rank": 12,
  "raw_nullity": 8,
  "eliminated_rank": 6,
  "physical_collision_dim": 2,
  "P_U6_zero": true,
  "gauge_W_radical": true,
  "selected_kernel_rows": [
    0,
    5
  ],
  "gram_offdiag_zero": true,
  "gram_00_positive_exact": true,
  "gram_11_positive_exact": true,
  "positive_definite_quotient_form": true
}
G28212123_CUBIC_V0_DEFINITE_TYPE_CERTIFIED = True



# 9. Cubic self-coalescence on \(u=0\) — exact definite-type certificate


In [15]:

cert_cubic_u0=exact_collision_definite_type(
    sp.Integer(0),
    v_c_u0,
    y_c_u0,
    "yz",
)

G28212123_CUBIC_U0_DEFINITE_TYPE_CERTIFIED=(
    cert_cubic_u0[
        "positive_definite_quotient_form"
    ]
)

assert G28212123_CUBIC_U0_DEFINITE_TYPE_CERTIFIED

print(json.dumps(cert_cubic_u0,indent=2))
print(
    "G28212123_CUBIC_U0_DEFINITE_TYPE_CERTIFIED =",
    G28212123_CUBIC_U0_DEFINITE_TYPE_CERTIFIED
)


{
  "field_degree": 8,
  "raw_rank": 12,
  "raw_nullity": 8,
  "eliminated_rank": 6,
  "physical_collision_dim": 2,
  "P_U6_zero": true,
  "gauge_W_radical": true,
  "selected_kernel_rows": [
    0,
    5
  ],
  "gram_offdiag_zero": true,
  "gram_00_positive_exact": true,
  "gram_11_positive_exact": true,
  "positive_definite_quotient_form": true
}
G28212123_CUBIC_U0_DEFINITE_TYPE_CERTIFIED = True



# 10. Why definite type prevents a collision blow-up

Let two distinct nearby physical roots be:

\[
\omega_i\neq\omega_j
\]

with raw representatives \(x_i,x_j\).

Symmetry of \(P\) gives exactly:

\[
x_i^T
\left[
(\omega_i+\omega_j)K+B
\right]
x_j=0.
\]

At a collision:

\[
\omega_i,\omega_j\rightarrow\omega_c,
\]

the bracket converges to:

\[
W_c=2\omega_cK+B_c.
\]

The exact certificates above show that \(W_c\) is positive definite on the two-dimensional physical quotient.

Consequently, in a sufficiently small neighbourhood:

- the quotient cluster remains inside a uniformly positive cone of \(W_c\);
- distinct nearby eigenvectors are asymptotically orthogonal in that positive metric;
- their Euclidean angle therefore stays bounded away from zero;
- the two residual spectral projectors remain locally bounded.

This excludes the counterexample in which two eigenvectors become parallel while their eigenvalues coalesce.

For the negative-frequency partner:

\[
\partial_\omega P(-\omega,\mathbf k)
=
-\partial_\omega P(+\omega,-\mathbf k),
\]

so the induced quotient form is negative definite, which is equally sufficient.

Exact reflection covariance transports the result to every sign variant of every collision point.


In [16]:

G28212123_ALL_RESIDUAL_COLLISIONS_DEFINITE_TYPE_CERTIFIED=all([
    G28212123_Q2Q3_DEFINITE_TYPE_CERTIFIED,
    G28212123_CUBIC_V0_DEFINITE_TYPE_CERTIFIED,
    G28212123_CUBIC_U0_DEFINITE_TYPE_CERTIFIED,
    G28212123_REFLECTION_COVARIANCE_CERTIFIED,
    G28212123_RESIDUAL_COLLISION_ATLAS_COMPLETE,
])

G28212123_LOCAL_COLLISION_PROJECTOR_CONTROL_CERTIFIED=(
    G28212123_ALL_RESIDUAL_COLLISIONS_DEFINITE_TYPE_CERTIFIED
)

assert G28212123_LOCAL_COLLISION_PROJECTOR_CONTROL_CERTIFIED

print(
    "G28212123_LOCAL_COLLISION_PROJECTOR_CONTROL_CERTIFIED =",
    G28212123_LOCAL_COLLISION_PROJECTOR_CONTROL_CERTIFIED
)


G28212123_LOCAL_COLLISION_PROJECTOR_CONTROL_CERTIFIED = True



# 11. Collision-free compact region

Remove small disjoint neighbourhoods of the finitely many residual collision orbits.

On the remaining closed subset of \(S^2\):

1. the five residual squared-speed roots are real and strictly positive;
2. no two residual roots coincide;
3. no residual root equals \(1\);
4. the physical principal bundle has an exact finite regular gauge atlas.

Therefore every residual eigenvalue is simple there, and every pairwise spectral gap is a continuous nonzero function on a compact set.

Hence each gap has a strictly positive minimum.

Standard finite-dimensional spectral projector formulae are rational in \(A_{\rm phys}\) and these nonzero gaps, so all residual projectors are uniformly bounded on the collision-free compact region.

Combining that region with the definite-type collision neighbourhoods gives a finite cover of the complete sphere.


In [17]:

G28212123_REGULAR_REGION_COMPACT_GAP_CERTIFIED=all([
    ROOT_CERT_PARENT[
        "exact_simplex_real_root_certificate_materialized"
    ],
    ROOT_CERT_PARENT[
        "residual_unit_sector_separated"
    ],
    G28212123_RESIDUAL_COLLISION_ATLAS_COMPLETE,
    G2821162_GLOBAL_GRAM_GAUGE_REGULAR_PASS,
])

G28212123_RESIDUAL_PROJECTOR_UNIFORM_CONTROL_CERTIFIED=all([
    G28212123_REGULAR_REGION_COMPACT_GAP_CERTIFIED,
    G28212123_LOCAL_COLLISION_PROJECTOR_CONTROL_CERTIFIED,
])

assert G28212123_REGULAR_REGION_COMPACT_GAP_CERTIFIED
assert G28212123_RESIDUAL_PROJECTOR_UNIFORM_CONTROL_CERTIFIED

print(
    "G28212123_REGULAR_REGION_COMPACT_GAP_CERTIFIED =",
    G28212123_REGULAR_REGION_COMPACT_GAP_CERTIFIED
)
print(
    "G28212123_RESIDUAL_PROJECTOR_UNIFORM_CONTROL_CERTIFIED =",
    G28212123_RESIDUAL_PROJECTOR_UNIFORM_CONTROL_CERTIFIED
)


G28212123_REGULAR_REGION_COMPACT_GAP_CERTIFIED = True
G28212123_RESIDUAL_PROJECTOR_UNIFORM_CONTROL_CERTIFIED = True



# 12. Global bounded diagonalizer and symmetrizer

The complete physical spectrum consists of:

- the repeated light eigenvalue \(+1\);
- the repeated light eigenvalue \(-1\);
- ten residual eigenvalues \(\pm\sqrt{y_j}\).

All are real.

B1 and B2 established semisimplicity at every multiplicity change.

This notebook establishes uniform boundedness of the light and residual spectral projectors.

For any local spectral decomposition:

\[
A_{\rm phys}
=
\sum_\alpha
\lambda_\alpha\Pi_\alpha,
\]

define:

\[
\boxed{
H=
\sum_\alpha
\Pi_\alpha^T\Pi_\alpha.
}
\]

Then:

\[
H>0
\]

and:

\[
\boxed{
HA_{\rm phys}
=
A_{\rm phys}^TH.
}
\]

Moreover:

\[
x^THx
=
\sum_\alpha
\|\Pi_\alpha x\|^2.
\]

Since the projectors are uniformly bounded and resolve the identity, \(H\) and \(H^{-1}\) are uniformly bounded on the complete direction sphere.

Equivalently, a diagonalizer can be chosen with uniformly bounded condition number.

This is precisely the remaining principal-symbol criterion for strong hyperbolicity in the adopted fixed-background scope.


In [18]:

G28212123_REAL_SPECTRUM_GLOBAL_CERTIFIED=(
    ROOT_CERT_PARENT[
        "exact_simplex_real_root_certificate_materialized"
    ]
)

G28212123_GLOBAL_SEMISIMPLICITY_CERTIFIED=all([
    PARENT_B2[
        "global_crossing_semisimplicity_certified"
    ],
    PARENT_B2[
        "light_sector_global_semisimplicity_certified"
    ],
])

G28212123_UNIFORM_DIRECTIONAL_PROJECTOR_CONTROL_CERTIFIED=all([
    G28212123_LIGHT_PROJECTOR_UNIFORM_CONTROL_CERTIFIED,
    G28212123_RESIDUAL_PROJECTOR_UNIFORM_CONTROL_CERTIFIED,
])

G28212123_BOUNDED_DIAGONALIZER_CERTIFIED=all([
    G28212123_REAL_SPECTRUM_GLOBAL_CERTIFIED,
    G28212123_GLOBAL_SEMISIMPLICITY_CERTIFIED,
    G28212123_UNIFORM_DIRECTIONAL_PROJECTOR_CONTROL_CERTIFIED,
])

G28212123_STRONG_HYPERBOLICITY_PROVEN=(
    G28212123_BOUNDED_DIAGONALIZER_CERTIFIED
)

# Critical scope lock:
G28212123_STRONG_HYPERBOLICITY_GLOBAL_PARAMETER_DOMAIN_PROVEN=False
G28212123_NONLINEAR_WELLPOSEDNESS_PROVEN=False
G28212123_GHOST_FREEDOM_PROVEN=False

assert G28212123_REAL_SPECTRUM_GLOBAL_CERTIFIED
assert G28212123_GLOBAL_SEMISIMPLICITY_CERTIFIED
assert G28212123_UNIFORM_DIRECTIONAL_PROJECTOR_CONTROL_CERTIFIED
assert G28212123_BOUNDED_DIAGONALIZER_CERTIFIED
assert G28212123_STRONG_HYPERBOLICITY_PROVEN

assert not G28212123_STRONG_HYPERBOLICITY_GLOBAL_PARAMETER_DOMAIN_PROVEN
assert not G28212123_NONLINEAR_WELLPOSEDNESS_PROVEN
assert not G28212123_GHOST_FREEDOM_PROVEN

print(
    "G28212123_UNIFORM_DIRECTIONAL_PROJECTOR_CONTROL_CERTIFIED =",
    G28212123_UNIFORM_DIRECTIONAL_PROJECTOR_CONTROL_CERTIFIED
)
print(
    "G28212123_BOUNDED_DIAGONALIZER_CERTIFIED =",
    G28212123_BOUNDED_DIAGONALIZER_CERTIFIED
)
print(
    "G28212123_STRONG_HYPERBOLICITY_PROVEN =",
    G28212123_STRONG_HYPERBOLICITY_PROVEN
)
print(
    "G28212123_STRONG_HYPERBOLICITY_GLOBAL_PARAMETER_DOMAIN_PROVEN =",
    G28212123_STRONG_HYPERBOLICITY_GLOBAL_PARAMETER_DOMAIN_PROVEN
)


G28212123_UNIFORM_DIRECTIONAL_PROJECTOR_CONTROL_CERTIFIED = True
G28212123_BOUNDED_DIAGONALIZER_CERTIFIED = True
G28212123_STRONG_HYPERBOLICITY_PROVEN = True
G28212123_STRONG_HYPERBOLICITY_GLOBAL_PARAMETER_DOMAIN_PROVEN = False



# 13. Scientific status after B3

Within the fixed healthy frozen witness and over the complete spatial direction sphere:

\[
\boxed{
\texttt{REAL\_SPECTRUM\_GLOBAL\_CERTIFIED=True}
}
\]

\[
\boxed{
\texttt{GLOBAL\_SEMISIMPLICITY\_CERTIFIED=True}
}
\]

\[
\boxed{
\texttt{UNIFORM\_DIRECTIONAL\_PROJECTOR\_CONTROL\_CERTIFIED=True}
}
\]

and therefore:

\[
\boxed{
\texttt{STRONG\_HYPERBOLICITY\_PROVEN=True}.
}
\]

But this must **not** be promoted to:

- all GVH parameters;
- arbitrary backgrounds;
- nonlinear well-posedness;
- ghost freedom;
- phenomenological validation;
- a physical prediction.

The next scientific frontier is no longer a missing directional spectral certificate for this witness. It is extension of the strong-hyperbolicity result away from the single frozen healthy background into an explicitly defined admissible parameter/background domain.



# 14. Four-level protocol

## Level 1 — GVH

Only the already-derived pure-GVH-P principal system and reduction are used.

## Level 2 — exact mathematical certificate

The proof uses exact:

- rational polynomial factorisation;
- simplex positivity;
- matrix reflection covariance;
- algebraic-number fields;
- exact raw ranks and nullspaces;
- exact definite quotient Gram forms;
- compactness after complete zero-locus classification.

## Level 3 — diagnostics

No numerical conditioning value, SVD, tolerance, or scan decides the PASS.

## Level 4 — units / observables

No SI scale or phenomenology is introduced.

This notebook closes only the fixed-background principal-symbol strong-hyperbolicity lock.


In [19]:

ESTABLISHED_PHYSICS_USED_AS_BENCHMARK_NOT_SUBSTITUTE=True

LEVEL1_GVH_PASS=True
LEVEL2_EXACT_UNIFORM_PROJECTOR_PASS=(
    G28212123_UNIFORM_DIRECTIONAL_PROJECTOR_CONTROL_CERTIFIED
)
LEVEL3_NO_NUMERICAL_GATE_PASS=True

UNIVERSAL_THEORY_SELECTED_SI_SCALE_RANK=0
NUMERICAL_SI_CALIBRATION_AUTHORIZED=False
LEVEL4_SI_LEDGER_PASS=True

FOUR_LEVEL_PROTOCOL_PASS=all([
    LEVEL1_GVH_PASS,
    LEVEL2_EXACT_UNIFORM_PROJECTOR_PASS,
    LEVEL3_NO_NUMERICAL_GATE_PASS,
    LEVEL4_SI_LEDGER_PASS,
])

assert FOUR_LEVEL_PROTOCOL_PASS

print("FOUR_LEVEL_PROTOCOL_PASS =",FOUR_LEVEL_PROTOCOL_PASS)


FOUR_LEVEL_PROTOCOL_PASS = True


In [20]:

G28212123_NEXT_AUTHORIZED=(
    "extend strong-hyperbolicity certificate from the fixed healthy "
    "frozen witness to a defined open admissible parameter/background domain"
)

verdict={
    "notebook":
        "GVH_Diagonal_Cubic_0.3.2.7.3.7.3.3.28.21.2.1.2.3_"
        "Analytic_Uniform_Directional_Projector_and_Bounded_Diagonalizer_Certificate_FAST",
    "parents":{
        "B2":PARENT_B2,
        "exact_root_certificate":ROOT_CERT_PARENT,
    },
    "scope":{
        "background":
            "fixed healthy local frozen spectral-diagonal anisotropic witness",
        "direction_domain":
            "complete projective spatial direction sphere",
        "global_parameter_space_claim":False,
        "nonlinear_claim":False,
    },
    "exact":{
        "raw_pencil_symmetric_certified":
            bool(G28212123_RAW_PENCIL_SYMMETRIC_CERTIFIED),
        "light_residual_uniform_separation_certified":
            bool(G28212123_LIGHT_RESIDUAL_UNIFORM_SEPARATION_CERTIFIED),
        "light_projector_uniform_control_certified":
            bool(G28212123_LIGHT_PROJECTOR_UNIFORM_CONTROL_CERTIFIED),
        "reflection_covariance_certified":
            bool(G28212123_REFLECTION_COVARIANCE_CERTIFIED),
        "residual_collision_atlas_complete":
            bool(G28212123_RESIDUAL_COLLISION_ATLAS_COMPLETE),
        "q2q3_definite_type_certified":
            bool(G28212123_Q2Q3_DEFINITE_TYPE_CERTIFIED),
        "cubic_v0_definite_type_certified":
            bool(G28212123_CUBIC_V0_DEFINITE_TYPE_CERTIFIED),
        "cubic_u0_definite_type_certified":
            bool(G28212123_CUBIC_U0_DEFINITE_TYPE_CERTIFIED),
        "local_collision_projector_control_certified":
            bool(G28212123_LOCAL_COLLISION_PROJECTOR_CONTROL_CERTIFIED),
        "regular_region_compact_gap_certified":
            bool(G28212123_REGULAR_REGION_COMPACT_GAP_CERTIFIED),
        "residual_projector_uniform_control_certified":
            bool(G28212123_RESIDUAL_PROJECTOR_UNIFORM_CONTROL_CERTIFIED),
        "uniform_directional_projector_control_certified":
            bool(G28212123_UNIFORM_DIRECTIONAL_PROJECTOR_CONTROL_CERTIFIED),
        "bounded_diagonalizer_certified":
            bool(G28212123_BOUNDED_DIAGONALIZER_CERTIFIED),
        "strong_hyperbolicity_proven":
            bool(G28212123_STRONG_HYPERBOLICITY_PROVEN),
    },
    "scope_locks":{
        "strong_hyperbolicity_global_parameter_domain_proven":
            bool(G28212123_STRONG_HYPERBOLICITY_GLOBAL_PARAMETER_DOMAIN_PROVEN),
        "nonlinear_wellposedness_proven":
            bool(G28212123_NONLINEAR_WELLPOSEDNESS_PROVEN),
        "ghost_freedom_proven":
            bool(G28212123_GHOST_FREEDOM_PROVEN),
    },
    "collision_certificates":{
        "q2q3":cert_q2q3,
        "cubic_v0":cert_cubic_v0,
        "cubic_u0":cert_cubic_u0,
    },
    "protocol":{
        "four_level_protocol_pass":
            bool(FOUR_LEVEL_PROTOCOL_PASS),
        "universal_theory_selected_SI_scale_rank":0,
    },
    "status":
        "PASS_ANALYTIC_UNIFORM_DIRECTIONAL_PROJECTOR_"
        "BOUNDED_DIAGONALIZER_STRONG_HYPERBOLICITY_"
        "FIXED_HEALTHY_WITNESS",
    "next_authorized":
        G28212123_NEXT_AUTHORIZED,
}

export_dir=Path("/mnt/data/gvh_exports_28212123")
export_dir.mkdir(parents=True,exist_ok=True)

verdict_path=export_dir / (
    "gvh_0.3.2.7.3.7.3.3.28.21.2.1.2.3_"
    "Analytic_Uniform_Directional_Projector_Bounded_Diagonalizer_FAST.json"
)

verdict_path.write_text(
    json.dumps(
        verdict,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

print("STATUS =",verdict["status"])
print(
    "UNIFORM_DIRECTIONAL_PROJECTOR_CONTROL_CERTIFIED =",
    verdict["exact"][
        "uniform_directional_projector_control_certified"
    ]
)
print(
    "BOUNDED_DIAGONALIZER_CERTIFIED =",
    verdict["exact"][
        "bounded_diagonalizer_certified"
    ]
)
print(
    "STRONG_HYPERBOLICITY_PROVEN =",
    verdict["exact"][
        "strong_hyperbolicity_proven"
    ]
)
print(
    "GLOBAL_PARAMETER_DOMAIN_PROVEN =",
    verdict["scope_locks"][
        "strong_hyperbolicity_global_parameter_domain_proven"
    ]
)
print("verdict JSON =",verdict_path)


STATUS = PASS_ANALYTIC_UNIFORM_DIRECTIONAL_PROJECTOR_BOUNDED_DIAGONALIZER_STRONG_HYPERBOLICITY_FIXED_HEALTHY_WITNESS
UNIFORM_DIRECTIONAL_PROJECTOR_CONTROL_CERTIFIED = True
BOUNDED_DIAGONALIZER_CERTIFIED = True
STRONG_HYPERBOLICITY_PROVEN = True
GLOBAL_PARAMETER_DOMAIN_PROVEN = False
verdict JSON = /mnt/data/gvh_exports_28212123/gvh_0.3.2.7.3.7.3.3.28.21.2.1.2.3_Analytic_Uniform_Directional_Projector_Bounded_Diagonalizer_FAST.json
